In [489]:
from google.colab import drive
import sqlite3
import pandas as pd

In [490]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [491]:
import os

BASE_DIR = '/content/drive/MyDrive/Colab_Notebooks'

db_path = os.path.join(BASE_DIR, "ML_Jazz", "data", "wjazzd.db")

conn = sqlite3.connect(db_path)

### wjazzd_query

In [492]:
pd.set_option('future.no_silent_downcasting', True)

In [493]:
def get_solo(melid: int, # all note+beat+meta
             conn: sqlite3.Connection) -> pd.DataFrame:
    query = f"""
    SELECT
    -- notes (melody)
    m.eventid,
    m.bar,                               -- join key
    m.pitch,                              -- pitch
    m.duration    AS note_duration,       -- duration
    m.beat,                               -- beat_pos (coarse) join key
    m.tatum,                              -- phrase segmentation: reset to 1 at phrase boundaries
    m.beatdur     AS beat_duration,       --
    m.onset       AS note_onset,          -- swing_ratio = (beat2_onset - beat1_onset) / (beat3_onset - beat1_onset)
    m.loud_max,                           -- vel
    m.loud_relpos,                        -- note_shape
    m.f0_mod,                             -- future expressive token (vibrato/fall-off/bend/slide)


    -- solo meatdata
    s.instrument,
    s.tempoclass,
    s.key,
    s.chorus_count,
    s.style,
    s.avgtempo

FROM melody m

JOIN beats b
    ON b.melid = m.melid
    AND b.bar = m.bar
    AND b.beat = m.beat

JOIN solo_info s
    ON s.melid = m.melid

WHERE m.melid = {melid}
ORDER BY m.eventid
"""

    return pd.read_sql_query(query, conn)

In [494]:
def get_beats(melid, conn): #  all bars, all beats, including silent
  query = f"""SELECT
      b.bar,
      b.beat,
      b.onset      AS beat_onset,
      b.chord,
      b.form,
      b.chorus_id


  FROM beats b
  WHERE b.melid = {melid}
  ORDER BY b.onset"""

  df = pd.read_sql_query(query, conn)
  df= df.set_index(["bar", "beat"])
  return df

In [495]:
def get_phrases(melid: int, # phrase spans
                conn: sqlite3.Connection) -> pd.DataFrame:
    query = f"""
    SELECT start, end, value
    FROM sections
    WHERE melid = {melid}
    AND type = 'PHRASE'
    ORDER BY start
    """
    return  pd.read_sql_query(query, conn)


In [496]:
def get_valid_ids(conn: sqlite3.Connection) -> list[int]:
    query = """
    SELECT melid
    FROM solo_info
    WHERE instrument IN (
          'tp', 'as',
           'ts', 'ss')
    AND signature = '4/4'
    """
    return pd.read_sql_query(query, conn)["melid"].tolist()

### chord_fn

In [497]:
def extract_root(symbol):
  roots = ["A#", "Ab", "A",
           "G#", "Gb", "G",
           "F#", "Fb", "F",
           "E#", "Eb", "E",
           "D#", "Db", "D",
           "C#", "Cb", "C",
           "B#", "Bb", "B", "NC"]
  for root in roots:
      if symbol.startswith(root):
          return root, symbol[len(root):]
  return "UNKNOWN", symbol

def extract_shape(symbol):
  shapes = [
    ("sus", "SUS"), ("sus2", "SUS"), ("sus4", "SUS"),
    ("maj", "MAJ"), ("Maj", "MAJ"), ("j",  "MAJ"), ("M",  "MAJ"),
    ("-",   "MIN"), ("m",   "MIN"),
    ("dim", "DIM"), ("o",   "DIM"),
    ("aug", "AUG"), ("+",   "AUG")]
  for marker, shape in shapes:
    if symbol.startswith(marker):
      if symbol == "j":
        return shape, symbol
      else:
        return shape, symbol[len(marker):]
  return "DOM", symbol          # no marker = dominant

def extract_7th(symbol):
  lis = [("j7", "MAJ7"),
         ("7", "MIN7")]
  for sig, name in lis:
    if symbol.startswith(sig):
      return name, symbol[len(sig):]
  return "NONE", symbol

def extract_alteration(symbol):
    lis = [("b5", "b5"),
          ("#5", "#5")]
    for sig, name in lis:
      if symbol.startswith(sig):
        return name, symbol[len(sig):]
    return "NONE", symbol

def extract_9th(symbol):
  lis = [
      ("9b", "b9"),
         ("9#", "#9"),
         ("b9", "b9"),
         ("#9", "#9"),
         ("9", "NAT9"),]
  for sig, name in lis:
    if symbol.startswith(sig):
      return name, symbol[len(sig):]
  return "UNKNOWN", symbol

def extract_11th(symbol):
  lis = [
      ("11b", "b11"),
      ("11#", "#11"),
      ("b11", "b11"),
      ("#11", "#11"),
      ("11", "NAT11")]
  for sig, name in lis:
    if symbol.startswith(sig):
      return name, symbol[len(sig):]
  return "UNKNOWN", symbol

def extract_13th(symbol):
  lis = [
      ("13b", "b13"),
      ("13#", "#13"),
      ("b13", "b13"),
      ("#13", "#13"),
      ("13", "NAT13")]
  for sig, name in lis:
    if symbol.startswith(sig):
      return name, symbol[len(sig):]
  return "UNKNOWN", symbol

def parse_chord(symbol: str) -> list[int]:
  """
  parse_chord("Abm7b59b1113")
  ['Ab', 'MIN', 'MIN7', 'b5', 'b9', 'NAT11', 'NAT13']
  """
  if not symbol:
        return ["UNKNOWN"] * 7
  root, symbol    = extract_root(symbol)
  shape, symbol   = extract_shape(symbol)
  ten7, symbol   = extract_7th(symbol) # hope spy chord is not lurking in
  alter, symbol   = extract_alteration(symbol)
  if symbol.startswith("alt"):
    return [root, shape, ten7, "b9", "#9", "#11", "b13"]
  ten9, symbol    = extract_9th(symbol)
  ten11, symbol   = extract_11th(symbol)
  ten13, symbol   = extract_13th(symbol)
  return [root, shape, ten7, alter, ten9, ten11, ten13]

#### EARLY/MID/PEAK/LATE

In [498]:
def classify_chorus(notes):
  """
   # Intensity =
   note_density(number of notes in chorus / number of bars in chorus)
   × tension_ratio (number of notes outside {R, 3, 5, 7} / total notes in chorus)
  """
  chorus_ids = notes["chorus_id"].dropna().unique()
  num_choruses = len(chorus_ids)
  if num_choruses == 0:
      return {}
  intensities = {}

  for chorus_idx in chorus_ids:
      chorus_block = notes[notes["chorus_id"] == chorus_idx]
      notes_in_cho = len(chorus_block)

      if notes_in_cho == 0:
        intensities[chorus_idx] = 0
        continue

      bars_in_cho = len(chorus_block["bar"].unique())
      bars_in_cho = max(1, bars_in_cho)

      note_density = notes_in_cho / bars_in_cho

      notes_outside_in_cho = build_chorus_map(chorus_block, chorus_idx)
      tension_ratio = notes_outside_in_cho / notes_in_cho

      intensity = note_density * tension_ratio
      intensities[chorus_idx] = intensity
  peak_chorus_id = max(intensities, key = intensities.get)
  chorus_tags = {}
  for i, chorus_idx in enumerate(chorus_ids):
      if chorus_idx == peak_chorus_id:
          chorus_tags[chorus_idx] = "PEAK"
      elif i == 0:
          chorus_tags[chorus_idx] = "EARLY"
      elif i == num_choruses - 1:
          chorus_tags[chorus_idx] = "LATE"
      else:
          chorus_tags[chorus_idx] = "MID"

  return chorus_tags


def build_chorus_map(chorus_block, chorus_idx):
  notes_outside = 0
  active_chord = ""
  fallback_key = chorus_block["key"].iloc[0].replace("-", "") if "key" in chorus_block.columns else "C"

  for i in range(len(chorus_block)):
    row = chorus_block.iloc[i]
    pitch = row["pitch"]
    current_chord = row.get("chord", "")
    if pd.notna(current_chord) and current_chord != "":
        active_chord = current_chord

    chord_to_parse = active_chord if active_chord != "" else fallback_key

    core_tones = chord_to_notes(chord_to_parse)

    if (pitch % 12) not in core_tones:
        notes_outside += 1

    return notes_outside

In [499]:
def chord_to_notes(input: str):
  """
  # ['Ab', 'MIN', 'MIN7', 'b5', 'b9', 'NAT11', 'NAT13']
  chord_to_notes(input)
  return [1.0, 2.0, 3.0, 5.0, 6.0, 8.0, 9.0, 11.0]
  """
  chord = parse_chord(input)
  chord_num_list = []
  root_num = {
      "C":  0.0, "B#": 0.0,
      "C#": 1.0, "Db": 1.0,
      "D":  2.0,
      "D#": 3.0, "Eb": 3.0,
      "E":  4.0, "Fb": 4.0,
      "F":  5.0, "E#": 5.0,
      "F#": 6.0, "Gb": 6.0,
      "G":  7.0,
      "G#": 8.0, "Ab": 8.0,
      "A":  9.0,
      "A#": 10.0, "Bb": 10.0,
      "B":  11.0, "Cb": 11.0}
  root_val = root_num.get(chord[0], 0.0)

  shape_num = {"SUS": [5, 7], "MAJ": [4, 7], "MIN": [3, 7], "DIM": [3, 6], "AUG": [4, 8], "DOM": [4, 7]}
  ten7_num  = {"MAJ7": [11], "MIN7": [10], "NONE": []}
  alter_num = {"b5": [6], "#5": [8], "NONE": []}
  ten9_num  = {"b9": [1], "NAT9": [2], "#9": [3], "UNKNOWN": []}
  ten11_num = {"b11": [4], "NAT11": [5], "#11": [6], "UNKNOWN": []}
  ten13_num = {"b13": [8], "NAT13": [9], "#13": [10], "UNKNOWN": []}

  relative_intervals = [0] # 0 represents the root itself
  dictionaries = [shape_num, ten7_num, alter_num, ten9_num, ten11_num, ten13_num]

  for vocab_dict, token in zip(dictionaries, chord[1:]):
      relative_intervals.extend(vocab_dict.get(token, []))

  chord_num_list = []
  for interval in relative_intervals:
      absolute_pc = (root_val + interval) % 12
      chord_num_list.append(absolute_pc)

  return sorted(list(set(chord_num_list)))

In [500]:
ROOT_VOCAB  = {"C":0,
               "C#":1, "Db":1,
               "D":2,
               "D#":3, "Eb":3,
               "E":4, "Fb":4,
               "F":5,
               "F#":6, "Gb":6,
               "G":7,
               "G#":8, "Ab":8,
               "A":9,
               "A#":10, "Bb":10,
               "B":11, "Cb": 11,
               "NC":12, "UNKNOWN":13}

SHAPE_VOCAB = {"MAJ":0, "MIN":1, "DOM":2, "DIM":3, "AUG":4, "SUS":5, "UNKNOWN":6}
TEN7_VOCAB  = {"NONE":0, "MIN7":1, "MAJ7":2, "UNKNOWN":3}
ALTER_VOCAB = {"NONE":0, "b5":1, "#5":2, "UNKNOWN":3, "b9": 4} # for alt b9 #9
TEN9_VOCAB  = {"NONE":0, "NAT9":1, "b9":2, "#9":3, "UNKNOWN":4}
TEN11_VOCAB = {"NONE":0, "NAT11":1, "b11":2, "#11":3, "UNKNOWN":4}
TEN13_VOCAB = {"NONE":0, "NAT13":1, "b13":2, "#13":3, "UNKNOWN":4}

INSTRUMENT_VOCAB = {
    "tp":0,
    "as":1,
    "ts":2,
    "ss":3,
    "UNKNOWN":4,
                    }
TEMPO_VOCAB = {
    "SLOW":0,
    "MEDIUM SLOW":1,
    "MEDIUM":2,
    "MEDIUM UP":3,
    "UP":4
}
SECTION_VOCAB = {
    "A1": 0,
    "A2": 0,
    "A3": 0,
    "B1": 1,
    "B2": 1,
    "C1": 2,
    "C2": 2,
    "C3": 2,
    "I1": 3,
    "I2": 3,
    "I3": 3,
    "I4": 3,
    "": 4,
}

CHORUS_VOCAB = {
    "EARLY":   0,
    "MID":     1,
    "PEAK":    2,
    "LATE":    3,
    "UNKNOWN": 4
}

def build_bar_header(bar_row, chorus_tags):

    chord_tokens = parse_chord(bar_row["chord"])   # → 7 strings
    chord_ids = [
        ROOT_VOCAB[chord_tokens[0]],
        SHAPE_VOCAB[chord_tokens[1]],
        TEN7_VOCAB[chord_tokens[2]],
        ALTER_VOCAB[chord_tokens[3]],
        TEN9_VOCAB[chord_tokens[4]],
        TEN11_VOCAB[chord_tokens[5]],
        TEN13_VOCAB[chord_tokens[6]]
    ]
    instrument = INSTRUMENT_VOCAB[bar_row["instrument"]]
    tempo_id      = TEMPO_VOCAB.get(bar_row["tempoclass"], 5) # USED for stratify
    section_id    = SECTION_VOCAB.get(bar_row["form"], 4)

    chorus_label = chorus_tags.get(bar_row["chorus_id"], "UNKNOWN")  # EARLY/MID/PEAK/LATE classification
    chorus_id    = CHORUS_VOCAB.get(chorus_label, 4)

    return chord_ids + [tempo_id, section_id, chorus_id, instrument]

#### Chord carry-forward pass

In [501]:
def chord_carry_forward(notes):
  notes = notes.copy()
  last_known = "UNKOWN"
  for i, row in notes.iterrows():
    if notes.at[i, "chord"].strip() == "":
        notes.at[i, "chord"] = last_known
    else:
        last_known = notes.at[i, "chord"]

  return notes

### classify_chorus(notes)/ bar groupby loop

In [502]:
# merged[~merged["is_rest"]]["loud_max"].describe()
# # merged[~merged["is_rest"]]["loud_max"].hist(bins=50)

In [503]:
VEL_VOCAB = {
    "pp":   0,
    "mp":   1,
    "mf":   2,
    "f":    3,
    "ff":   4,
    "REST": 5
}

def compute_velocity_bins(merged):
    loud_vals = merged.loc[~merged["is_rest"], "loud_max"]
    loud_vals = loud_vals[loud_vals > loud_vals.quantile(0.01)]
    return loud_vals.quantile([0.2, 0.4, 0.6, 0.8]).values

def assign_velocity(loud_max, bins):
    if loud_max == 0.0:
        return VEL_VOCAB["REST"]
    labels = ["pp", "mp", "mf", "f", "ff"]
    for i, threshold in enumerate(bins):
        if loud_max < threshold:
            return VEL_VOCAB[labels[i]]
    return VEL_VOCAB["ff"]

In [504]:
NOTESHAPE_VOCAB = {
    "accent-release": 0,
    "decay":          1,
    "flat":           2,
    "swell":          3,
    "REST":           4
}

BEATPOS_VOCAB = {i: i for i in range(16)}   # 0–15 direct

DURATION_VOCAB = {i: i for i in range(32)}  # 0–31 sixteenth units, cap at 2 bars

def assign_noteshape(loud_relpos):
    if   loud_relpos < 0.2: return NOTESHAPE_VOCAB["accent-release"]
    elif loud_relpos < 0.4: return NOTESHAPE_VOCAB["decay"]
    elif loud_relpos < 0.6: return NOTESHAPE_VOCAB["flat"]
    else:                   return NOTESHAPE_VOCAB["swell"]

def quantise_duration(note_duration, beat_duration):
    sixteenths = note_duration / beat_duration * 4
    return min(int(round(sixteenths)), 31)

def compute_beatpos(beat, note_onset, beat_onset, beat_duration):
    position_in_beat = (note_onset - beat_onset) / beat_duration
    position_in_beat = max(position_in_beat, 0.0)        # clamp negative laid-back
    subdivision      = min(int(position_in_beat * 4), 3) # 0–3 within beat
    return (beat - 1) * 4 + subdivision                  # 0–15 across bar

def build_note_tuple(row, bar_header, vel_bins):

    if row["is_rest"]:
        return {
            "bar":       int(row["bar"]),
            "beatpos":   compute_beatpos(row["beat"], row["beat_onset"],
                                         row["beat_onset"], row["beat_duration"]),
            "swing":     row["swing_ratio"],   # precomputed column
            "duration":  quantise_duration(row["note_duration"], row["beat_duration"]),
            "pitch":     0,                    # REST_PITCH
            "interval":  -1,                   # placeholder
            "tension":   -1,                   # placeholder
            "phrase_role": -1,                 # placeholder
            "vel":       VEL_VOCAB["REST"],
            "noteshape": NOTESHAPE_VOCAB["REST"],
            "is_rest":   True
        }

    return {
        "bar":       int(row["bar"]),
        "beatpos":   compute_beatpos(row["beat"], row["note_onset"],
                                     row["beat_onset"], row["beat_duration"]),
        "swing":     row["swing_ratio"],       # precomputed column
        "duration":  quantise_duration(row["note_duration"], row["beat_duration"]),
        "pitch":     int(round(row["pitch"])), # range check happens before this
        "interval":  -1,                       # placeholder — needs bar_header
        "tension":   -1,                       # placeholder — needs bar_header
        "phrase_role": -1,                     # placeholder — needs phrases df
        "vel":       assign_velocity(row["loud_max"], vel_bins),
        "noteshape": assign_noteshape(row["loud_relpos"]),
        "is_rest":   False
    }

#### Compute Swing

In [505]:
SWING_VOCAB = {
    "SW0":      0,   # 0.50–0.54  straight - > up, fast tempo
    "SW1":      1,   # 0.54–0.58  slight lilt
    "SW2":      2,   # 0.58–0.62  medium swing
    "SW3":      3,   # 0.62–0.67  full swing
    "SW4":      4,   # 0.67–0.72  heavy swing
    "SW5":      5,   # 0.72+      extreme
    "SW_UNKNOWN": 6
}

def discretize_swing(ratio):
    if   ratio < 0.54: return SWING_VOCAB["SW0"]
    elif ratio < 0.58: return SWING_VOCAB["SW1"]
    elif ratio < 0.62: return SWING_VOCAB["SW2"]
    elif ratio < 0.67: return SWING_VOCAB["SW3"]
    elif ratio < 0.72: return SWING_VOCAB["SW4"]
    else:              return SWING_VOCAB["SW5"]

def compute_swing_prepass(merged):
    beat_grid = (merged[["bar", "beat", "beat_onset"]]
                 .drop_duplicates()
                 .sort_values(["bar", "beat"])
                 .reset_index(drop=True))

    swing_map = {}   # (bar, beat) → SW token

    for i in range(len(beat_grid) - 1):
        curr = beat_grid.iloc[i]
        next_ = beat_grid.iloc[i + 1]

        # only pair odd beats with their following even beat
        # beat 1→2, beat 3→4 within same bar
        is_valid_pair = (
            curr["beat"] in [1, 3] and
            next_["beat"] == curr["beat"] + 1 and
            next_["bar"] == curr["bar"]
        )

        if not is_valid_pair:
            swing_map[(curr["bar"], curr["beat"])] = SWING_VOCAB["SW_UNKNOWN"]
            continue

        # find the beat after the pair for denominator
        after = beat_grid.iloc[i + 2] if i + 2 < len(beat_grid) else None

        if after is None:
            swing_map[(curr["bar"], curr["beat"])] = SWING_VOCAB["SW_UNKNOWN"]
            swing_map[(next_["bar"], next_["beat"])] = SWING_VOCAB["SW_UNKNOWN"]
            continue

        pair_duration = after["beat_onset"] - curr["beat_onset"]
        long_duration = next_["beat_onset"] - curr["beat_onset"]
        ratio = long_duration / pair_duration

        sw_token = discretize_swing(ratio)
        swing_map[(curr["bar"], curr["beat"])]  = sw_token
        swing_map[(next_["bar"], next_["beat"])] = sw_token   # both beats share same SW

    # map back onto merged
    merged["swing_ratio"] = merged.apply(
        lambda r: swing_map.get((r["bar"], r["beat"]), SWING_VOCAB["SW_UNKNOWN"]),
        axis=1
    )
    return merged

In [506]:
def treat_nans(merged):
    solo_cols = ["instrument", "tempoclass", "key", "chorus_count", "style", "avgtempo"]
    merged[solo_cols] = merged[solo_cols].ffill().bfill()

    merged["eventid"]       = merged["eventid"].fillna(-1).astype(int)
    merged["is_rest"] =  merged["eventid"] == -1
    merged["tatum"]         = merged["tatum"].fillna(-1).astype(int)
    merged["pitch"]         = merged["pitch"].fillna(0).astype(int)
    merged["note_onset"]    = merged["note_onset"].fillna(merged["beat_onset"])

    rest_mask = merged["is_rest"]
    merged.loc[rest_mask, "note_duration"] = (
        merged["beat_onset"].shift(-1) - merged["beat_onset"]
    )[rest_mask].fillna(0)
    merged["loud_max"]      = merged["loud_max"].fillna(0.0)
    merged["loud_relpos"]   = merged["loud_relpos"].fillna(0.0)
    merged["f0_mod"]        = merged["f0_mod"].fillna(0).replace("", 0)
    merged["beat_duration"] = merged["beat_duration"].ffill().bfill()
    return merged

#### Compute Interval

In [507]:
INTERVAL_VOCAB = {
    0:  "R",     # root
    1:  "b2",    # rare in jazz, chromatic approach
    2:  "2",     # major 2nd / 9th
    3:  "b3",    # minor 3rd
    4:  "3",     # major 3rd
    5:  "4",     # perfect 4th / 11th
    6:  "b5",    # tritone
    7:  "5",     # perfect 5th
    8:  "#5",    # augmented 5th / b13
    9:  "6",     # major 6th / 13th
    10: "b7",    # minor 7th
    11: "7",     # major 7th
    12: "UNKNOWN"
}

def compute_interval(pitch, bar_header):
    root_pitch = bar_header[0]
    if root_pitch == 13:
        return 12
    note_pitch = int(round(pitch)) % 12
    interval = (note_pitch - root_pitch) % 12
    return interval

#### compute tension role

In [508]:
TENSION_VOCAB = {
    "CHORD_TONE":  0,
    "EXTENSION":   1,
    "ALTERATION":  2,
    "APPROACH":    3,
    "OUTSIDE":     4,
    "REST":        5,
    "UNKNOWN":     6
}

def get_chord_tones(bar_header):
    shape  = bar_header[1]
    ten7   = bar_header[2]
    alter  = bar_header[3]

    base = {
        SHAPE_VOCAB["MAJ"]: {0, 4, 7},
        SHAPE_VOCAB["MIN"]: {0, 3, 7},
        SHAPE_VOCAB["DOM"]: {0, 4, 7},
        SHAPE_VOCAB["DIM"]: {0, 3, 6},
        SHAPE_VOCAB["AUG"]: {0, 4, 8},
        SHAPE_VOCAB["SUS"]: {0, 5, 7},
    }.get(shape, {0})

    if ten7 == TEN7_VOCAB["MIN7"]: base.add(10)
    if ten7 == TEN7_VOCAB["MAJ7"]: base.add(11)

    return base

def get_extensions(bar_header):
    ext = set()
    ten9_map  = {TEN9_VOCAB["NAT9"]:2,  TEN9_VOCAB["b9"]:1,   TEN9_VOCAB["#9"]:3}
    ten11_map = {TEN11_VOCAB["NAT11"]:5, TEN11_VOCAB["#11"]:6}
    ten13_map = {TEN13_VOCAB["NAT13"]:9, TEN13_VOCAB["b13"]:8}

    if bar_header[4] in ten9_map:  ext.add(ten9_map[bar_header[4]])
    if bar_header[5] in ten11_map: ext.add(ten11_map[bar_header[5]])
    if bar_header[6] in ten13_map: ext.add(ten13_map[bar_header[6]])
    return ext

def get_alterations(bar_header):
    alt_map = {
        ALTER_VOCAB["b5"]: 6,
        ALTER_VOCAB["#5"]: 8,
    }
    return {alt_map[bar_header[3]]} if bar_header[3] in alt_map else set()

def compute_tension_role(interval, bar_header, next_interval):
    if interval == -1:
        return TENSION_VOCAB["REST"]
    if interval == 12:
        return TENSION_VOCAB["UNKNOWN"]

    chord_tones = get_chord_tones(bar_header)
    extensions  = get_extensions(bar_header)
    alterations = get_alterations(bar_header)

    if interval in chord_tones:
        return TENSION_VOCAB["CHORD_TONE"]

    if interval in extensions:
        return TENSION_VOCAB["EXTENSION"]

    if interval in alterations:
        return TENSION_VOCAB["ALTERATION"]

    # APPROACH — semitone from a chord tone, resolves to chord tone on next note
    if next_interval not in (-1, 12): # requires one-row look-ahead
        semitone_away = abs(interval - next_interval) == 1 or \
                        abs(interval - next_interval) == 11  # wrap-around (e.g. 11→0)
        if semitone_away and next_interval in chord_tones:
            return TENSION_VOCAB["APPROACH"]

    return TENSION_VOCAB["OUTSIDE"]

In [509]:
PHRASEROLE_VOCAB = {
    "START": 0,
    "CONT":  1,
    "PEAK":  2,
    "LAND":  3,
    "REST":  4,
    "UNKNOWN": 5
}

def compute_phrase_roles(merged, phrases):
  """
  map eventid to phrase role before the note loop
  # assume no inter-phrase gaps
  """
  role_map = {}

  for _, phrase in phrases.iterrows():
      phrase_notes = merged[
          (merged["eventid"] >= phrase["start"]) &
          (merged["eventid"] <= phrase["end"]) &
          (~merged["is_rest"])
      ]

      if phrase_notes.empty:
          continue

      peak_eventid  = phrase_notes.loc[phrase_notes["pitch"].idxmax(), "eventid"]
      start_eventid = phrase["start"]
      end_eventid   = phrase["end"]

      for _, row in phrase_notes.iterrows():
          eid = row["eventid"]
          if eid == start_eventid:
              role_map[eid] = PHRASEROLE_VOCAB["START"]
          elif eid == end_eventid:
              role_map[eid] = PHRASEROLE_VOCAB["LAND"]
          elif eid == peak_eventid:
              role_map[eid] = PHRASEROLE_VOCAB["PEAK"]
          else:
              role_map[eid] = PHRASEROLE_VOCAB["CONT"]

  # map back onto merged
  merged["phrase_role"] = merged.apply(
      lambda r: PHRASEROLE_VOCAB["REST"] if r["is_rest"]
                else role_map.get(int(r["eventid"]), PHRASEROLE_VOCAB["UNKNOWN"]),
      axis=1
  )
  return merged

### Build Note Tuple

In [510]:
def build_note_tuple(row, vel_bins):
    """
    row -> all precomputed columns
    vel_bins -> solo-level quantile range
    """
    if row["is_rest"]:
        return {
            "bar":        int(row["bar"]),
            "beatpos":    compute_beatpos(row["beat"], row["beat_onset"],
                                          row["beat_onset"], row["beat_duration"]),
            "swing":      row["swing_ratio"],
            "duration":   quantise_duration(row["note_duration"], row["beat_duration"]),
            "pitch":      0,
            "interval":   -1,
            "octave": -1,
            "tension":    TENSION_VOCAB["REST"],
            "phrase_role":PHRASEROLE_VOCAB["REST"],
            "vel":        VEL_VOCAB["REST"],
            "noteshape":  NOTESHAPE_VOCAB["REST"],
            "is_rest":    True
        }

    return {
        "bar":        int(row["bar"]),
        "beatpos":    compute_beatpos(row["beat"], row["note_onset"],
                                      row["beat_onset"], row["beat_duration"]),
        "swing":      row["swing_ratio"],
        "duration":   quantise_duration(row["note_duration"], row["beat_duration"]),
        "pitch":      int(round(row["pitch"])),
        "interval":   int(row["interval"]),
        "octave": int(round(row["pitch"])) // 12,
        "tension":    int(row["tension_role"]),
        "phrase_role":int(row["phrase_role"]),
        "vel":        assign_velocity(row["loud_max"], vel_bins),
        "noteshape":  assign_noteshape(row["loud_relpos"]),
        "is_rest":    False
    }


### Full Assembly

In [511]:
def assemble(token_seq, bar_headers):
  seq = []
  current_bar = -1

  for note in token_seq:
    bar_num = note["bar"]
    if bar_num != current_bar:
      seq.extend(bar_headers[bar_num])
      current_bar = bar_num

    note_tuple =[
        note["beatpos"],
        note["swing"],
        note["duration"],
        note["pitch"],
        note["interval"],
        note["octave"],
        note["tension"],
        note["phrase_role"],
        note["vel"],
        note["noteshape"]
    ]
    seq.extend(note_tuple)
    return seq

In [512]:
import torch

SPECIAL_TOKENS = {
    "PAD": 1000,
    "SOLO_END": 1001,
    "END": 1002  # before padding
}
def process_and_save_corpus(conn, output_dir):
    success_count = 0
    error_count = 0
    melids = get_valid_ids(conn) # 343 ids available

    for melid in melids:
        try:
            ####### QUERIES #########
            notes   = get_solo(melid, conn)
            beats   = get_beats(melid, conn)
            phrases = get_phrases(melid, conn)

            ####### MERGE #########
            merged = beats.merge(notes, on=["bar", "beat"], how="left")
            merged = treat_nans(merged)
            merged  = chord_carry_forward(merged)

            ####### SOLO INFO #########
            chorus_tags = classify_chorus(merged)
            vel_bins = compute_velocity_bins(merged)   # once per solo
            merged = compute_swing_prepass(merged)

            ####### BAR INFO #########
            bar_headers = {}
            for bar_num, bar_group in merged.groupby("bar"):
                bar_row = bar_group.iloc[0]
                bar_headers[bar_num] = build_bar_header(bar_row, chorus_tags)


            ####### NOTE INFO #########
            merged["interval"] = merged.apply(
            lambda r: -1 if r["is_rest"] else compute_interval(
                r["pitch"],
                bar_headers[r["bar"]]),
            axis=1
            )
            merged["next_interval"] = merged["interval"].shift(-1).fillna(-1).astype(int)
            merged.loc[merged["is_rest"], "interval"]      = -1
            merged.loc[merged["is_rest"], "next_interval"] = -1

            merged["tension_role"] = merged.apply(
                lambda r: TENSION_VOCAB["REST"] if r["is_rest"]
                else compute_tension_role(
                    r["interval"],
                    bar_headers[r["bar"]],
                    r["next_interval"]
                ),
                axis = 1
                )
            merged = compute_phrase_roles(merged, phrases)

            ####### TOKENIZE NOTES #######
            token_seq = []
            for _, row in merged.iterrows():
              note = build_note_tuple(row, vel_bins)
              token_seq.append(note)
        # [bar, beatpos, swing, duration, pitch, interval, octave, tension, phrase_role, vel, noteshape, is_rest]

            seq_1D  = assemble(token_seq, bar_headers)
            seq_1D.append(SPECIAL_TOKENS["SOLO_END"])

            seq_tensor = torch.tensor(seq_1D, dtype = torch.long)

            save_path = os.path.join(output_dir, f"melid_{melid}.pt")
            torch.save(seq_tensor, save_path)

            success_count += 1

        except Exception as e:
            print(f"Fail on {melid}: {e}")
            error_count += 1
    print(f"saved to {output_dir}")


OUTPUT_DIR = "/content/drive/MyDrive/Colab_Notebooks/ML_Jazz/data/processed"
process_and_save_corpus(conn, OUTPUT_DIR)

saved to /content/drive/MyDrive/Colab_Notebooks/ML_Jazz/data/processed


inter-phrase silence               → REST phrase token, duration from onset gap  
fully silent bars                  → REST token, detected from beats EXCEPT melody

rest_duration = next_phrase_first_note_onset - last_phrase_last_note_offset